# 💳 Modélisation Prédictive & Détection de Fraude Bancaire (Bloc 3 - RNCP 41993)

Ce notebook documente l'entraînement supervisé, le feature engineering géographique et temporel, ainsi que l'export du modèle pour le projet **Automatic Fraud Detection**.

### 🎯 Objectifs Métier & RNCP :
1. **Chargement des données historiques :** `fraudTest.csv` (555 719 transactions bancaires).
2. **Feature Engineering :** Distance orthodromique de Haversine (porteur - marchand), extraction de l'heure, âge.
3. **Modélisation Supervisée :** `HistGradientBoostingClassifier` avec pondération équilibrée des classes.
4. **Évaluation :** ROC-AUC > 0.98, Rappel Fraude > 91%.
5. **Sérialisation :** Export vers `fraud_model.joblib` et `model_metadata.json` pour le microservice FastAPI.

In [ ]:
import os
import sys
import time
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, recall_score, precision_score, f1_score

print("✅ Bibliothèques chargées avec succès.")

## 1. Chargement des Données Historiques

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
CSV_PATH = os.path.join(BASE_DIR, '..', 'fraudTest.csv')
if not os.path.exists(CSV_PATH):
    CSV_PATH = 'fraudTest.csv'

# Chargement d'un sous-ensemble représentatif
df = pd.read_csv(CSV_PATH, nrows=200000)
print(f"Total transactions chargées : {len(df)}")
print(f"Nombre de fraudes réelles : {df['is_fraud'].sum()} ({df['is_fraud'].mean()*100:.2f}%)")
df.head(3)

## 2. Feature Engineering Géographique & Temporel

In [ ]:
def haversine_np(lon1, lat1, lon2, lat2):
    """Calcule la distance orthodromique en km entre porteur et commerçant."""
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return 6367 * 2 * np.arcsin(np.sqrt(a))

X = pd.DataFrame()
X['amt'] = df['amt'].astype(float)
X['city_pop'] = df['city_pop'].fillna(1000).astype(float)
X['distance_km'] = haversine_np(df['long'], df['lat'], df['merch_long'], df['merch_lat'])

time_col = 'trans_date_trans_time' if 'trans_date_trans_time' in df.columns else 'current_time'
if time_col in df.columns:
    X['hour'] = pd.to_datetime(df[time_col], errors='coerce').dt.hour.fillna(12).astype(int)
else:
    X['hour'] = 12

if 'dob' in df.columns:
    X['age'] = (2026 - pd.to_datetime(df['dob'], errors='coerce').dt.year).fillna(40).astype(int)
else:
    X['age'] = 40

y = df['is_fraud']
print("Statistiques descriptives des features calculées :")
X.describe()

## 3. Entraînement de l'Algorithme HistGradientBoosting

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

params = {
    "class_weight": "balanced",
    "max_iter": 120,
    "learning_rate": 0.08,
    "max_depth": 6,
    "random_state": 42
}

model = HistGradientBoostingClassifier(**params)
model.fit(X_train, y_train)
print("✅ Modèle entraîné avec succès.")

## 4. Évaluation des Performances

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"• ROC-AUC Score : {roc_auc:.4f}")
print(f"• Rappel Fraude : {recall:.4f} ({recall*100:.1f}% des fraudes interceptées)")
print(f"• Précision     : {precision:.4f}")
print(f"• F1-Score      : {f1:.4f}")
print("\nRapport complet :\n", classification_report(y_test, y_pred, digits=4))

## 5. Export des Poids & Métadonnées pour l'API

In [ ]:
output_model_path = os.path.join(BASE_DIR, 'fraud_model.joblib')
output_meta_path = os.path.join(BASE_DIR, 'model_metadata.json')

joblib.dump(model, output_model_path)

metadata = {
    "model_type": "HistGradientBoostingClassifier",
    "features": list(X.columns),
    "parameters": params,
    "metrics": {
        "roc_auc": round(float(roc_auc), 4),
        "recall": round(float(recall), 4),
        "precision": round(float(precision), 4),
        "f1": round(float(f1), 4)
    },
    "trained_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

with open(output_meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Modèle sauvegardé dans : {output_model_path}")
print(f"✅ Métadonnées sauvegardées dans : {output_meta_path}")